# Try Visual Jev on your own image

This notebook loads the 40k Visual Jev vision LoRA and runs one spatial question. Enable a GPU runtime for faster inference, then upload an image you choose. The upload goes to your Colab runtime; this notebook contains no sample photos.

The generic Hugging Face PEFT snippet uses `AutoModelForCausalLM`, which loads only the text backbone. This notebook loads `Qwen3_5ForConditionalGeneration` so the image is processed.

In [ ]:
%pip -q install 'transformers>=5.6,<6' 'peft>=0.21.2' pillow accelerate

In [ ]:
import torch
from google.colab import files
from PIL import Image
from peft import PeftModel
from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration

base_id = 'Qwen/Qwen3.5-0.8B-Base'
adapter_id = 'harshnandwana/visual-jev-budget20-qwen35-0.8b-lora'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if device == 'cuda' else torch.float32
processor = AutoProcessor.from_pretrained(base_id)
base = Qwen3_5ForConditionalGeneration.from_pretrained(base_id, dtype=dtype).to(device)
model = PeftModel.from_pretrained(base, adapter_id).eval()
print('Loaded', adapter_id, 'on', device)

In [ ]:
uploaded = files.upload()
image_path = next(iter(uploaded))
with Image.open(image_path) as source:
    photo = source.convert('RGB')
    photo.thumbnail((512, 512))
question = 'Is the person to the left of the backpack?'  # Edit for your image
prompt = question + '\nChoices: YES, NO, UNKNOWN. Answer with one choice only.'
messages = [{'role': 'user', 'content': [
    {'type': 'image', 'image': photo}, {'type': 'text', 'text': prompt}
]}]
inputs = processor.apply_chat_template(
    messages, chat_template=processor.tokenizer.chat_template,
    tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors='pt',
).to(device)
with torch.inference_mode():
    tokens = model.generate(**inputs, max_new_tokens=32, do_sample=False)
answer = processor.batch_decode(
    tokens[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True
)[0].strip()
print('Answer:', answer)